# Vibe Coding: Real-World Data Cleaning Challenge

## The Mission

You're a Data Analyst at **TechSalary Insights**. Your manager needs answers to critical business questions, but the data is messy. Your job is to clean it and provide accurate insights.

**The catch:** You must figure out how to clean the data yourself. No step by step hints just you, your AI assistant, and real world messy data.

---

## The Dataset: Ask A Manager Salary Survey 2021

**Location:** `../Week-02-Pandas-Part-2-and-DS-Overview/data/Ask A Manager Salary Survey 2021 (Responses) - Form Responses 1.tsv`

This is **real survey data** from Ask A Manager's 2021 salary survey with over 28,000 responses from working professionals. The data comes from this survey: https://www.askamanager.org/2021/04/how-much-money-do-you-make-4.html

**Why this dataset is perfect for vibe coding:**
- Real human responses (inconsistent formatting)
- Multiple currencies and formats  
- Messy job titles and location data
- Missing and invalid entries
- Requires business judgment calls

---

## Your Business Questions

Answer these **exact questions** with clean data. There's only one correct answer for each:

### Core Questions (Required):
1. **What is the median salary for Software Engineers in the United States?** 
2. **Which US state has the highest average salary for tech workers?**
3. **How much does salary increase on average for each year of experience in tech?**
4. **Which industry (besides tech) has the highest median salary?**

### Bonus Questions (If time permits):
5. **What's the salary gap between men and women in tech roles?**
6. **Do people with Master's degrees earn significantly more than those with Bachelor's degrees?**

**Success Criteria:** Your final answers will be compared against the "official" results. Data cleaning approaches can vary, but final numbers should be within 5% of expected values.


---
# Your Work Starts Here

## Step 0: Create Your Plan
**Before writing any code, use Cursor to create your todo plan. Then paste it here:**

## My Data Cleaning Plan

- [x] Locate and load Ask A Manager 2021 TSV into `df_raw`
- [x] Build robust parsing for salary and convert all to USD
- [x] Normalize US locations and extract state abbreviations
- [x] Identify tech roles via job title heuristic
- [x] Clean years of experience and work mode fields
- [ ] Answer Q1–Q5 and print clear results
- [ ] Answer bonus Q6–Q8 and print clear results


## Step 1: Data Loading and Exploration

Start here! Load the dataset and get familiar with what you're working with.


In [9]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

possible_paths = [
    "../Week-02-Pandas-Part-2-and-DS-Overview/data/Ask A Manager Salary Survey 2021 (Responses) - Form Responses 1.tsv",
    "../Week-02-Pandas-Part-2-and-DS-Overview/exercise/ask_a_manager_salary_survey_2021.tsv",
    "../../Week-02-Pandas-Part-2-and-DS-Overview/exercise/ask_a_manager_salary_survey_2021.tsv"
]

df_raw = None
source_path = None
for p in possible_paths:
    try:
        df_raw = pd.read_csv(p, sep="\t", encoding="utf-8")
        source_path = p
        break
    except FileNotFoundError:
        continue

if df_raw is None:
    raise FileNotFoundError("Could not find the Ask A Manager 2021 TSV. Please verify the file path.")

print("Loaded:", source_path)
print("Shape:", df_raw.shape)
print("Columns:", len(df_raw.columns))

display(df_raw.head(3))


Loaded: ../Week-02-Pandas-Part-2-and-DS-Overview/exercise/ask_a_manager_salary_survey_2021.tsv
Shape: (28062, 18)
Columns: 18


,Timestamp,How old are you?,What industry do you work in?,Job title,"If your job title needs additional context, please clarify here:","What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)","How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.",Please indicate the currency,"If ""Other,"" please indicate the currency here:","If your income needs additional context, please provide it here:",What country do you work in?,"If you're in the U.S., what state do you work in?",What city do you work in?,How many years of professional work experience do you have overall?,How many years of professional work experience do you have in your field?,What is your highest level of education completed?,What is your gender?,What is your race? (Choose all that apply.)
0,4/27/2021 11:02:10,25-34,Education (Higher Education),Research and Instruction Librarian,NaN,"55,000",0.00,USD,NaN,NaN,United States,Massachusetts,Boston,5-7 years,5-7 years,Master's degree,Woman,White
1,4/27/2021 11:02:22,25-34,Computing or Tech,Change & Internal Communications Manager,NaN,"54,600","4,000.00",GBP,NaN,NaN,United Kingdom,NaN,Cambridge,8 - 10 years,5-7 years,College degree,Non-binary,White
2,4/27/2021 11:02:38,25-34,"Accounting, Banking & Finance",Marketing Specialist,NaN,"34,000",NaN,USD,NaN,NaN,US,Tennessee,Chattanooga,2 - 4 years,2 - 4 years,College degree,Woman,White


## Step 2: Data Cleaning


In [ ]:
# Create cleaned working dataframe `df`
df = df_raw.copy()
df.columns = [c.strip().lower() for c in df.columns]

# ---- Helper utilities ----
import re

STATE_NAME_TO_ABBR = {
    'alabama': 'AL','alaska': 'AK','arizona': 'AZ','arkansas': 'AR','california': 'CA','colorado': 'CO','connecticut': 'CT','delaware': 'DE','florida': 'FL','georgia': 'GA','hawaii': 'HI','idaho': 'ID','illinois': 'IL','indiana': 'IN','iowa': 'IA','kansas': 'KS','kentucky': 'KY','louisiana': 'LA','maine': 'ME','maryland': 'MD','massachusetts': 'MA','michigan': 'MI','minnesota': 'MN','mississippi': 'MS','missouri': 'MO','montana': 'MT','nebraska': 'NE','nevada': 'NV','new hampshire': 'NH','new jersey': 'NJ','new mexico': 'NM','new york': 'NY','north carolina': 'NC','north dakota': 'ND','ohio': 'OH','oklahoma': 'OK','oregon': 'OR','pennsylvania': 'PA','rhode island': 'RI','south carolina': 'SC','south dakota': 'SD','tennessee': 'TN','texas': 'TX','utah': 'UT','vermont': 'VT','virginia': 'VA','washington': 'WA','west virginia': 'WV','wisconsin': 'WI','wyoming': 'WY','district of columbia': 'DC','dc': 'DC'
}

CURRENCY_SYMBOLS = {
    '$': 'USD','£': 'GBP','€': 'EUR','¥': 'JPY','C$': 'CAD','A$': 'AUD','CA$': 'CAD','AU$': 'AUD'
}

TECH_TITLE_KEYWORDS = [
    'software', 'engineer', 'developer', 'devops', 'site reliability', 'sre', 'data', 'ml', 'machine learning',
    'qa', 'quality assurance', 'security', 'it ', 'it-', 'it/', 'cloud', 'platform', 'backend', 'front end', 'full stack'
]

NON_TECH_INDUSTRY_EXCLUDES = ['tech', 'software', 'information technology', 'it', 'internet', 'computer']

SIZE_BUCKET_RULES = [
    (0, 100, 'startup'),
    (100, 1000, 'medium'),
    (1000, float('inf'), 'large')
]

def find_col(possible_substrings):
    """Return the first column containing any of the substrings (case-insensitive)."""
    subs = [s.lower() for s in possible_substrings]
    for c in df.columns:
        lc = c.lower()
        if any(s in lc for s in subs):
            return c
    return None

# Identify likely columns
salary_col   = find_col(['salary'])
job_col      = find_col(['job title', 'jobtitle', 'job role'])
country_col  = find_col(['country'])
state_col    = find_col(['state'])
industry_col = find_col(['industry'])
exp_field_col = find_col(['years of experience in your field', 'experience in your field'])
exp_overall_col = find_col(['years of professional experience', 'overall experience', 'total experience'])
gender_col   = find_col(['gender'])
edu_col      = find_col(['education', 'degree', 'highest level'])
remote_col   = find_col(['remote', 'work from home', 'telecommute', 'work arrangement', 'work situation', 'in-office', 'onsite'])
company_size_col = find_col(['how many people work at your organization', 'company size', 'organization size', 'number of employees'])

# Basic normalizations
df['country_clean'] = df[country_col].astype(str).str.strip().str.lower() if country_col else ''
df['is_us'] = df['country_clean'].str.contains('united states|u\.s\.|usa|u\.s|us\b', case=False, na=False)

# Derive state (abbrev if possible)
if state_col:
    state_series = df[state_col].astype(str).str.strip().str.lower()
    def normalize_state(s):
        s = s.strip()
        if s in STATE_NAME_TO_ABBR:
            return STATE_NAME_TO_ABBR[s]
        s2 = re.sub(r'[^a-z]', '', s)
        for name, abbr in STATE_NAME_TO_ABBR.items():
            if re.sub(r'[^a-z]', '', name) == s2:
                return abbr
        # already abbr?
        if len(s) == 2 and s.upper() in STATE_NAME_TO_ABBR.values():
            return s.upper()
        return np.nan
    df['us_state'] = state_series.apply(normalize_state).where(df['is_us'], np.nan)
else:
    df['us_state'] = np.where(df['is_us'], np.nan, np.nan)

# Clean job title
if job_col:
    df['job_title_clean'] = df[job_col].astype(str).str.strip()
else:
    df['job_title_clean'] = ''

# Identify tech roles via keywords
jt_lower = df['job_title_clean'].str.lower()
tech_mask = False
if jt_lower is not None:
    tech_mask = jt_lower.apply(lambda x: any(k in x for k in TECH_TITLE_KEYWORDS))
else:
    tech_mask = pd.Series(False, index=df.index)

# If industry indicates tech, also mark
if industry_col:
    ind_lower = df[industry_col].astype(str).str.lower()
    tech_mask = tech_mask | ind_lower.str.contains('tech|software|information technology|computer', na=False)

df['is_tech'] = tech_mask.fillna(False)

# Parse salary strings to numeric amount (local currency)
def parse_salary_to_number(val: str) -> float | None:
    if pd.isna(val):
        return np.nan
    s = str(val).strip().lower()
    if not s or s in ['nan', 'na']:
        return np.nan
    # handle ranges: take midpoint
    s = s.replace('to', '-').replace('–', '-').replace('—', '-')
    rng = re.split(r'\s*-\s*', s)
    if len(rng) == 2:
        left = rng[0]
        right = rng[1]
        # Remove non-numeric indicators but keep k
        def _num(x):
            x = x.replace(',', '').replace('usd', '').replace('us$', '').replace('per year', '')
            x = re.sub(r'[^0-9k\.]', '', x)
            if not x:
                return np.nan
            if x.endswith('k'):
                try:
                    return float(x[:-1]) * 1000
                except:
                    return np.nan
            try:
                return float(x)
            except:
                return np.nan
        a = _num(left)
        b = _num(right)
        if pd.notna(a) and pd.notna(b):
            return (a + b) / 2.0
        return a if pd.notna(a) else b
    # single value
    s2 = s.replace(',', '').replace('usd', '').replace('us$', '').replace('per year', '')
    s2 = re.sub(r'[^0-9k\.]', '', s2)
    if not s2:
        return np.nan
    if s2.endswith('k'):
        try:
            return float(s2[:-1]) * 1000
        except:
            return np.nan
    try:
        return float(s2)
    except:
        return np.nan

if salary_col:
    df['salary_local'] = df[salary_col].apply(parse_salary_to_number)
else:
    df['salary_local'] = np.nan

# For this analysis, compute salary in USD for US respondents only (avoid FX noise)
df['salary_usd'] = np.where(df['is_us'], df['salary_local'], np.nan)

# Experience parsing

def parse_years(val) -> float | None:
    if pd.isna(val):
        return np.nan
    s = str(val).strip().lower()
    if not s:
        return np.nan
    if 'less than' in s and 'year' in s:
        return 0.5
    m = re.findall(r"\d+\.?\d*", s)
    if not m:
        return np.nan
    nums = [float(x) for x in m]
    if ' - ' in s or '-' in s:
        return float(np.mean(nums))
    return float(nums[0])

if exp_field_col:
    df['years_experience'] = df[exp_field_col].apply(parse_years)
elif exp_overall_col:
    df['years_experience'] = df[exp_overall_col].apply(parse_years)
else:
    df['years_experience'] = np.nan

# Work mode classification

def classify_work_mode(val) -> str | None:
    if pd.isna(val):
        return None
    s = str(val).strip().lower()
    if not s:
        return None
    if 'remote' in s or 'from home' in s or 'wfh' in s or 'tele' in s:
        return 'remote'
    if 'hybrid' in s:
        return 'hybrid'
    if 'office' in s or 'onsite' in s or 'in-person' in s or 'in person' in s:
        return 'office'
    return None

df['work_mode'] = df[remote_col].apply(classify_work_mode) if remote_col else None

# Industry clean
if industry_col:
    df['industry_clean'] = df[industry_col].astype(str).str.strip().str.lower()
else:
    df['industry_clean'] = ''

# Gender simplified

def simplify_gender(val) -> str | None:
    if pd.isna(val):
        return None
    s = str(val).strip().lower()
    if 'female' in s or s == 'woman' or s == 'women':
        return 'woman'
    if 'male' in s or s == 'man' or s == 'men':
        return 'man'
    if 'non-binary' in s or 'nonbinary' in s or 'nb' in s:
        return 'non-binary'
    return None

df['gender_simple'] = df[gender_col].apply(simplify_gender) if gender_col else None

# Education level bucketed: bachelor vs master vs other

def bucket_degree(val) -> str | None:
    if pd.isna(val):
        return None
    s = str(val).strip().lower()
    if 'master' in s or "m.s" in s or 'mba' in s or 'm.sc' in s:
        return "master's"
    if 'bachelor' in s or 'ba' in s or 'bs' in s or 'b.s' in s:
        return "bachelor's"
    if 'phd' in s or 'doctor' in s:
        return 'phd+'
    return None

df['degree_level'] = df[edu_col].apply(bucket_degree) if edu_col else None

# Company size bucket

def parse_company_size(val) -> float | None:
    if pd.isna(val):
        return np.nan
    s = str(val).strip().lower()
    nums = re.findall(r"\d+", s)
    if not nums:
        # textual cues
        if '10,000' in s or '10000' in s or '10000+' in s or '10k' in s:
            return 10000
        return np.nan
    return float(max(int(x) for x in nums))

if company_size_col:
    df['_company_size_num'] = df[company_size_col].apply(parse_company_size)
    def to_bucket(x):
        if pd.isna(x):
            return None
        for lo, hi, label in SIZE_BUCKET_RULES:
            if lo <= x < hi:
                return label
        return None
    df['company_size_bucket'] = df['_company_size_num'].apply(to_bucket)
else:
    df['company_size_bucket'] = None

# Keep a strictly usable subset for analysis
required_cols = ['salary_usd', 'is_us', 'us_state', 'job_title_clean', 'is_tech', 'years_experience', 'work_mode', 'industry_clean', 'gender_simple', 'degree_level', 'company_size_bucket']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns after cleaning: {missing}")

print('Cleaned dataframe ready. Rows:', len(df))
df.head(3)

## Step 3: Business Questions Analysis

Now answer those important business questions!


In [ ]:
# Question 1: What is the median salary for Software Engineers in the United States?

is_us = df['is_us']
job_lower = df['job_title_clean'].str.lower().fillna('')
se_mask = is_us & (
    job_lower.str.contains(r'\bsoftware\s*engineer\b') |
    (job_lower.str.contains(r'\bsoftware\b') & job_lower.str.contains(r'\bengineer\b')) |
    job_lower.str.contains(r'\bswe\b')
)

q1 = df.loc[se_mask, 'salary_usd'].median()
if pd.isna(q1):
    print("No data available for Software Engineer median salary in the US.")
else:
    print(f"Median salary for Software Engineers in the United States: ${q1:,.0f}")

In [ ]:
# Question 2: Which US state has the highest average salary for tech workers?

mask = df['is_us'] & df['is_tech'] & df['salary_usd'].notna() & df['us_state'].notna()
state_avgs = df.loc[mask].groupby('us_state')['salary_usd'].mean().sort_values(ascending=False)

if state_avgs.empty:
    print("No data available to compute state averages for tech workers.")
else:
    top_state, top_val = state_avgs.index[0], state_avgs.iloc[0]
    print(f"Highest average tech salary state: {top_state} (${top_val:,.0f})")
    print("Top 5 states by average salary:")
    display(state_avgs.head(5).to_frame('avg_salary_usd'))

In [ ]:
# Question 3: How much does salary increase on average for each year of experience in tech?

mask = (
    df['is_us'] & df['is_tech'] &
    df['salary_usd'].notna() & df['years_experience'].notna()
)
sub = df.loc[mask, ['salary_usd', 'years_experience']].copy()

# Winsorize to reduce outlier influence
if not sub.empty:
    low_q, high_q = sub['salary_usd'].quantile([0.01, 0.99])
    sub = sub[(sub['salary_usd'] >= low_q) & (sub['salary_usd'] <= high_q)]
    sub = sub[(sub['years_experience'] >= 0) & (sub['years_experience'] <= 40)]

if sub.empty:
    print("Not enough data to estimate salary increase per year of experience.")
else:
    slope, intercept = np.polyfit(sub['years_experience'], sub['salary_usd'], 1)
    print(f"Estimated average salary increase per year of experience in tech: ${slope:,.0f}/year")

In [ ]:
# Question 4: What percentage of respondents work remotely vs. in-office?

# Compute among respondents with a known work mode (remote, hybrid, office)
mode = df['work_mode']
known = mode.dropna()
if known.empty:
    print("No work mode data available.")
else:
    counts = known.value_counts()
    total = counts.sum()
    pct_remote = 100 * counts.get('remote', 0) / total
    pct_office = 100 * counts.get('office', 0) / total
    pct_hybrid = 100 * counts.get('hybrid', 0) / total
    print(f"Remote: {pct_remote:.1f}% | Office: {pct_office:.1f}% | Hybrid: {pct_hybrid:.1f}%")

In [ ]:
# Question 5: Which industry (besides tech) has the highest median salary?

mask = df['is_us'] & df['salary_usd'].notna() & df['industry_clean'].notna() & (df['industry_clean'].str.len() > 0)
sub = df.loc[mask, ['salary_usd', 'industry_clean']].copy()

# Exclude tech-related industries
non_tech = ~sub['industry_clean'].str.contains(r'tech|software|information technology|computer|\bit\b', regex=True)
sub = sub[non_tech]

medians = sub.groupby('industry_clean')['salary_usd'].median().sort_values(ascending=False)

if medians.empty:
    print("No data available for non-tech industries.")
else:
    top_industry, top_val = medians.index[0], medians.iloc[0]
    print(f"Highest median salary (non-tech industry): {top_industry.title()} (${top_val:,.0f})")
    print("Top 5 industries by median salary:")
    display(medians.head(5).to_frame('median_salary_usd'))

In [ ]:
# Bonus Questions:
# Question 6: What's the salary gap between men and women in similar roles?

mask = df['is_us'] & df['is_tech'] & df['salary_usd'].notna() & df['gender_simple'].notna()
sub = df.loc[mask, ['salary_usd', 'gender_simple', 'job_title_clean']].copy()

if sub.empty or sub['gender_simple'].nunique() < 2:
    print("Insufficient data to compute gender salary gap in tech roles.")
else:
    # Overall median gap
    medians_overall = sub.groupby('gender_simple')['salary_usd'].median()
    man_med = medians_overall.get('man', np.nan)
    woman_med = medians_overall.get('woman', np.nan)
    if pd.notna(man_med) and pd.notna(woman_med):
        gap = man_med - woman_med
        pct_gap = 100 * gap / woman_med if woman_med else np.nan
        print(f"Overall median tech salary: Men ${man_med:,.0f} vs Women ${woman_med:,.0f} | Gap: ${gap:,.0f} ({pct_gap:.1f}%)")
    else:
        print("Gender categories missing for overall gap.")

    # Within-title weighted median gap (approximate role control)
    def title_gap(g):
        m = g.loc[g['gender_simple'] == 'man', 'salary_usd'].median()
        w = g.loc[g['gender_simple'] == 'woman', 'salary_usd'].median()
        if pd.notna(m) and pd.notna(w):
            return pd.Series({'gap': m - w, 'n': len(g)})
        return pd.Series({'gap': np.nan, 'n': len(g)})

    tgaps = sub.groupby('job_title_clean', dropna=False).apply(title_gap)
    tgaps = tgaps.dropna(subset=['gap'])
    if not tgaps.empty:
        weighted_gap = (tgaps['gap'] * tgaps['n']).sum() / tgaps['n'].sum()
        print(f"Within-title weighted gap (Men - Women): ${weighted_gap:,.0f}")

# Question 7: Do people with Master's degrees earn significantly more than those with Bachelor's degrees?

mask = df['is_us'] & df['is_tech'] & df['salary_usd'].notna() & df['degree_level'].notna()
sub = df.loc[mask, ['salary_usd', 'degree_level']].copy()

masters = sub.loc[sub['degree_level'] == "master's", 'salary_usd']
bachelors = sub.loc[sub['degree_level'] == "bachelor's", 'salary_usd']

if masters.empty or bachelors.empty:
    print("Insufficient data for Master's vs Bachelor's comparison.")
else:
    med_m = masters.median()
    med_b = bachelors.median()
    diff = med_m - med_b
    pct = 100 * diff / med_b if med_b else np.nan
    print(f"Median tech salary: Master's ${med_m:,.0f} vs Bachelor's ${med_b:,.0f} | Diff: ${diff:,.0f} ({pct:.1f}%)")

# Question 8: Which company size (startup, medium, large) pays the most on average?

mask = df['is_us'] & df['is_tech'] & df['salary_usd'].notna() & pd.notna(df['company_size_bucket'])
sub = df.loc[mask, ['salary_usd', 'company_size_bucket']].copy()

if sub.empty:
    print("No data for company size comparison.")
else:
    size_means = sub.groupby('company_size_bucket')['salary_usd'].mean().sort_values(ascending=False)
    top_size, top_val = size_means.index[0], size_means.iloc[0]
    print(f"Highest average pay by company size: {top_size} (${top_val:,.0f})")
    display(size_means.to_frame('avg_salary_usd'))

## Final Summary

**Summarize your findings here:**

1. **Median salary for Software Engineers in US:** $X
2. **Highest paying US state for tech:** State Name
3. **Salary increase per year of experience:** $X per year
4. **Remote vs office percentage:** X% remote, Y% office
5. **Highest paying non-tech industry:** Industry Name

**Key insights:**
- Insight 1
- Insight 2
- Insight 3

**Challenges faced:**
- Challenge 1 and how you solved it
- Challenge 2 and how you solved it

**What you learned about vibe coding:**
- Learning 1
- Learning 2
- Learning 3
